# Bronze Layer - Retail Sales Lakehouse

**Purpose:** Ingest raw sales data from landing zone with basic validation and standardization

---

### Input Sources:
* **Full Refresh:** `/landing/base/*.csv` - Initial baseline snapshot
* **Incremental:** `/updates/{process_date}/*.csv` - Daily incremental files

### Output Tables:
* `bronze_sales`: Full refresh baseline data
* `bronze_incremental`: Incremental update data

### Job Parameters:
* `full_refresh`: Boolean flag to determine load mode (true/false)
* `source_system`: Source system identifier for data lineage
* `process_date`: Date partition for incremental loads (required when full_refresh=false)

---

### Production Note:
In production, replace `spark.read.csv()` with Auto Loader (`spark.readStream.format("cloudFiles")`) for automatic incremental file discovery, checkpointing, and schema evolution. The Bronze transformation logic remains unchanged.

In [0]:
# Import required libraries
from pyspark.sql import functions as F

In [0]:
# Read job parameters
full_refresh = (
    dbutils.widgets.get("full_refresh")
    .strip()
    .lower() == "true"
)

source_system = dbutils.widgets.get("source_system").strip()

In [0]:
# Define paths for base and incremental loads
BASE_PATH = "/Workspace/Users/vinilg7@gmail.com/Retail-Sales-Lakehouse/landing/base/*.csv"

BRONZE_BASE_TABLE = "bronze_sales"

BRONZE_INCREMENTAL_TABLE = "bronze_incremental"

In [0]:
# Determine input path based on load mode
if full_refresh:

    print("Running FULL REFRESH")

    INPUT_PATH = BASE_PATH

else:

    process_date = dbutils.widgets.get("process_date").strip()

    if process_date == "":
        raise Exception(
            "process_date parameter is required for incremental loads."
        )

    print(f"Running INCREMENTAL LOAD : {process_date}")

    INPUT_PATH = (
        "/Workspace/Users/vinilg7@gmail.com/"
        f"Retail-Sales-Lakehouse/updates/{process_date}/*.csv"
    )

In [0]:
# Read CSV files from landing zone
sales_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(INPUT_PATH)
)

In [0]:
# Validate required columns are present
required_columns = {
    "transaction_id",
    "store_id",
    "customer_id",
    "product_category",
    "payment_method",
    "sales_amount",
    "transaction_time",
    "ingestion_time"
}

missing_columns = required_columns - set(sales_df.columns)

if missing_columns:

    raise Exception(
        f"Missing required columns: {missing_columns}"
    )

In [0]:
# Standardize datatypes for consistency
sales_df = (
    sales_df
        .withColumn(
            "sales_amount",
            F.col("sales_amount").cast("double")
        )
        .withColumn(
            "transaction_time",
            F.to_timestamp("transaction_time")
        )
        .withColumn(
            "ingestion_time",
            F.to_timestamp("ingestion_time")
        )
)

In [0]:
# Add metadata columns for lineage and tracking
sales_df = (
    sales_df
        .withColumn(
            "source_system",
            F.lit(source_system)
        )
        .withColumn(
            "bronze_loaded_at",
            F.current_timestamp()
        )
)

In [0]:
# Write to appropriate Bronze table based on load mode
if full_refresh:

    (
        sales_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(BRONZE_BASE_TABLE)
    )

    print("Bronze base table created.")

else:

    (
        sales_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(BRONZE_INCREMENTAL_TABLE)
    )

    print("Bronze incremental table created.")

In [0]:
# Display summary statistics and sample data
print("=" * 60)
print("Bronze Load Completed")
print("=" * 60)
print(f"Mode          : {'FULL REFRESH' if full_refresh else 'INCREMENTAL'}")
print(f"Rows Loaded   : {sales_df.count()}")
print(f"Source System : {source_system}")
print("=" * 60)

sales_df.show(truncate=False)